# Chan's Mean-Reversion Strategy — Yahoo Finance Backtest

A concise replication of Chan's cross-sectional reversal rule using **nine U.S. sector ETFs (2018–2025)**. This uses a different dataset from the book, so results need not match.

## 1. Imports and configuration

One-way transaction cost: **5 bps**. A two-session execution lag avoids assuming we trade at the close used to form the signal.

In [1]:
%pip install yfinance numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

TICKERS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]
START_DATE = "2018-01-01"
END_DATE = "2026-01-01"

# One-way cost: 5 bps = 0.05% per unit of traded notional
ONE_WAY_COST = 0.0005

# After-close signal; execute at the next close, then earn the following close-to-close return
EXECUTION_LAG = 2

pd.set_option("display.float_format", lambda x: f"{x:.6f}")


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Download Yahoo Finance data

Download adjusted closing prices. Keep dates with prices for all nine ETFs.

In [2]:
raw = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    group_by="column",
    progress=False,
    threads=False,
)

if raw.empty:
    raise RuntimeError("Yahoo Finance returned no data. Check your internet connection and try again.")

# Multi-ticker downloads typically use a MultiIndex: price field, then ticker
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    raise ValueError("Unexpected columns in the multi-ETF Yahoo Finance download.")

prices = prices.reindex(columns=TICKERS)
prices = prices.sort_index().dropna(how="any")

if prices.empty or len(prices) < 100:
    raise ValueError("Insufficient valid price observations for the backtest.")
if not (prices > 0).all().all():
    raise ValueError("Some prices are non-positive; inspect the data.")
if prices.index.has_duplicates:
    raise ValueError("Duplicate trading dates found in the downloaded data.")

print(f"Valid data: {len(prices)} trading days x {prices.shape[1]} ETFs")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
display(prices.head())

C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace

Valid data: 2011 trading days x 9 ETFs
Date range: 2018-01-02 to 2025-12-31


C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
C:\Users\22307\PycharmProjects\WelcomeScreen\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace

Ticker,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY
Date,,,,,,,,,
2018-01-02,25.780201,25.411856,23.716833,65.915077,29.802568,44.708279,19.858820,72.127945,46.081718
2018-01-03,25.960772,25.792423,23.844246,66.270119,30.051155,44.692463,19.702808,72.818092,46.293289
2018-01-04,26.187548,25.948122,24.065109,66.755035,30.203066,44.818989,19.539194,72.921646,46.445072
2018-01-05,26.397512,25.937740,24.133062,67.213997,30.520699,45.016663,19.531580,73.542755,46.813023
2018-01-08,26.435312,26.093428,24.099083,67.491066,30.635788,45.127365,19.714233,73.275330,46.868217


## 3. Daily returns

$$r_{i,t}=P_{i,t}/P_{i,t-1}-1.$$

In [3]:
dailyret = prices.pct_change(fill_method=None).iloc[1:]
assert dailyret.notna().all().all()
assert dailyret.shape[1] == len(TICKERS)
display(dailyret.head())

Ticker,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY
Date,,,,,,,,,
2018-01-03,0.007004,0.014976,0.005372,0.005386,0.008341,-0.000354,-0.007856,0.009568,0.004591
2018-01-04,0.008735,0.006037,0.009263,0.007317,0.005055,0.002831,-0.008304,0.001422,0.003279
2018-01-05,0.008018,-0.000400,0.002824,0.006875,0.010517,0.004410,-0.000390,0.008517,0.007922
2018-01-08,0.001432,0.006002,-0.001408,0.004122,0.003771,0.002459,0.009352,-0.003636,0.001179
2018-01-09,-0.001589,-0.002519,0.007755,0.006416,-0.002554,-0.001401,-0.009844,0.011773,0.001963


## 4. Chan's reversal signal

$$w_{i,t}=-\frac{r_{i,t}-\bar r_t}{\sum_j|r_{j,t}-\bar r_t|}.$$

Short yesterday's relative winners and buy yesterday's relative losers.

In [4]:
def make_chan_weights(returns: pd.DataFrame) -> pd.DataFrame:
    mean_return = returns.mean(axis=1)
    raw_signal = -returns.sub(mean_return, axis=0)
    gross_exposure = raw_signal.abs().sum(axis=1)
    weights = raw_signal.div(gross_exposure.replace(0, np.nan), axis=0)
    return weights.fillna(0.0)

signal_weights = make_chan_weights(dailyret)

# Check the defining properties on dates with a nonzero signal
active = signal_weights.abs().sum(axis=1) > 1e-12
np.testing.assert_allclose(
    signal_weights.loc[active].abs().sum(axis=1).to_numpy(),
    1.0, atol=1e-10
)
np.testing.assert_allclose(
    signal_weights.sum(axis=1).to_numpy(),
    0.0, atol=1e-10
)

display(signal_weights.head())

Ticker,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY
Date,,,,,,,,,
2018-01-03,-0.046092,-0.252666,-0.003802,-0.004167,-0.080735,0.144577,0.338986,-0.112537,0.016437
2018-01-04,-0.143767,-0.062528,-0.159643,-0.101080,-0.032982,0.033967,0.369162,0.076380,0.020491
2018-01-05,-0.088277,0.191953,0.084632,-0.050246,-0.171464,0.031808,0.191607,-0.104914,-0.085099
2018-01-08,0.044714,-0.132384,0.154759,-0.059531,-0.045917,0.004911,-0.262168,0.241104,0.054513
2018-01-09,0.057536,0.077356,-0.141585,-0.113047,0.078112,0.053541,0.233455,-0.227220,-0.018148


## 5. Backtest and transaction costs

Weights are lagged to avoid look-ahead. Gross return is the weighted portfolio return. Net return subtracts a **5 bps one-way cost** on traded notional, with turnover measured against drifted positions. This is a simplified historical backtest, not an execution simulation.

In [5]:
def run_backtest(
    returns: pd.DataFrame,
    target_weights: pd.DataFrame,
    one_way_cost: float = 0.0005,
    execution_lag: int = 2,
) -> pd.DataFrame:
    if execution_lag < 1:
        raise ValueError("execution_lag must be >= 1")
    if one_way_cost < 0:
        raise ValueError("one_way_cost cannot be negative")
    if not returns.index.equals(target_weights.index):
        raise ValueError("Returns and weights must use the same date index")

    # Weights held during period t, based on signals formed and executed earlier
    held = target_weights.shift(execution_lag).fillna(0.0)
    gross = (held * returns).sum(axis=1)

    # Weights drift with price changes by the close of day t
    portfolio_growth = 1.0 + gross
    if (portfolio_growth <= 0).any():
        raise ValueError("Portfolio value is non-positive; cannot calculate weight drift")
    drifted_close = held.mul(1.0 + returns).div(portfolio_growth, axis=0)

    # Compare current held weights with the prior close's drifted weights
    # Assume zero starting exposure so the initial rebalance incurs trading costs
    previous_drifted = drifted_close.shift(1).fillna(0.0)
    turnover = (held - previous_drifted).abs().sum(axis=1)

    costs = one_way_cost * turnover
    net = gross - costs

    result = pd.DataFrame({
        "gross_return": gross,
        "turnover": turnover,
        "transaction_cost": costs,
        "net_return": net,
    }, index=returns.index)

    # Drop the initial warm-up dates, retaining the first entry-cost observation
    result = result.iloc[execution_lag:]
    return result

backtest = run_backtest(
    returns=dailyret,
    target_weights=signal_weights,
    one_way_cost=ONE_WAY_COST,
    execution_lag=EXECUTION_LAG,
)

display(backtest.head())
print("Average daily turnover:", round(backtest["turnover"].mean(), 4))
print("Average daily transaction cost (bps):", round(10000 * backtest["transaction_cost"].mean(), 4))

,gross_return,turnover,transaction_cost,net_return
Date,,,,
2018-01-05,-0.001480,1.000000,0.000500,-0.001980
2018-01-08,0.002385,0.924284,0.000462,0.001923
2018-01-09,-0.002904,1.211860,0.000606,-0.003510
2018-01-10,0.003733,1.628139,0.000814,0.002919
2018-01-11,-0.001694,1.776754,0.000888,-0.002583


Average daily turnover: 1.3914
Average daily transaction cost (bps): 6.9568


## 6. Performance

Annualized Sharpe ratio: $\sqrt{252}\,\overline R/s(R)$. Compare gross and net performance.

In [6]:
def performance_metrics(returns: pd.Series) -> pd.Series:
    x = returns.dropna().astype(float)
    if len(x) < 2:
        raise ValueError("Too few observations to evaluate performance")

    vol = x.std(ddof=1)
    sharpe = np.sqrt(252) * x.mean() / vol if vol > 0 else np.nan

    wealth = (1 + x).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    annual_return = (wealth.iloc[-1] ** (252 / len(x)) - 1.0) if wealth.iloc[-1] > 0 else np.nan

    return pd.Series({
        "Annualized return": annual_return,
        "Annualized volatility": np.sqrt(252) * vol,
        "Annualized Sharpe": sharpe,
        "Maximum drawdown": drawdown.min(),
        "Mean daily return": x.mean(),
    })

summary = pd.DataFrame({
    "Gross": performance_metrics(backtest["gross_return"]),
    "Net (5 bps)": performance_metrics(backtest["net_return"]),
})
display(summary)

print(f"Gross Sharpe: {summary.loc['Annualized Sharpe', 'Gross']:.3f}")
print(f"Net Sharpe:   {summary.loc['Annualized Sharpe', 'Net (5 bps)']:.3f}")
print(f"Sharpe change after costs: {summary.loc['Annualized Sharpe', 'Net (5 bps)'] - summary.loc['Annualized Sharpe', 'Gross']:.3f}")

,Gross,Net (5 bps)
Annualized return,-0.015264,-0.173681
Annualized volatility,0.088308,0.088400
Annualized Sharpe,-0.130069,-2.113070
Maximum drawdown,-0.244034,-0.788596
Mean daily return,-0.000046,-0.000741


Gross Sharpe: -0.130
Net Sharpe:   -2.113
Sharpe change after costs: -1.983


## Conclusions

1. **Chan's one-day cross-sectional mean-reversion strategy did not work in this Yahoo Finance sample:** across these nine sector ETFs during 2018–2025, the observed gross Sharpe was approximately **−0.13**. This result does not imply that mean reversion never works for other assets or periods.
2. **Transaction costs have a substantial adverse impact:** frequent rebalancing makes net performance markedly worse than gross performance. The table above reports the impact for the modeled 5 bps one-way cost.